# Feature Engineering: Creating Better Inputs for ML Models

Feature engineering is the art of transforming raw data into features that better represent the underlying problem. Often more impactful than model selection.


In [ ]:
# Colab setup
import sys, os, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    for REPO in ['/content/drive/MyDrive/data-science-mastery', '/content/data-science-mastery']:
        if os.path.isdir(REPO):
            os.chdir(REPO)
            break
    print(f'Working dir: {os.getcwd()}')


## 1. Load Data


In [ ]:
df = pd.read_csv('Data/house_pricing.csv')
df.head()


## 2. Numerical Feature Engineering


In [ ]:
df_fe = df.copy()

# 2a. Polynomial features (interactions)
df_fe['size_bedroom'] = df_fe['size_sqft'] * df_fe['bedrooms']
df_fe['size_bath'] = df_fe['size_sqft'] * df_fe['bathrooms']
df_fe['rooms_total'] = df_fe['bedrooms'] + df_fe['bathrooms']

# 2b. Ratio features
df_fe['rooms_per_sqft'] = df_fe['rooms_total'] / df_fe['size_sqft']
df_fe['price_per_sqft'] = df_fe['price'] / df_fe['size_sqft']

# 2c. Binning / discretization
df_fe['property_age_group'] = pd.cut(df_fe['age_years'], bins=[0, 5, 20, 50, 100], labels=['New', 'Recent', 'Old', 'Very Old'])

# 2d. Log transform for skewed targets
df_fe['log_price'] = np.log1p(df_fe['price'])
df_fe[['price', 'log_price']].describe()


## 3. Categorical Feature Engineering


In [ ]:
# 3a. Frequency encoding
freq_enc = df_fe['location'].value_counts() / len(df_fe)
df_fe['location_freq'] = df_fe['location'].map(freq_enc)

# 3b. Target encoding (mean encoding)
target_mean = df_fe.groupby('location')['price'].mean()
df_fe['location_target_enc'] = df_fe['location'].map(target_mean)

# 3c. One-hot encoding
df_fe = pd.get_dummies(df_fe, columns=['location', 'property_age_group'], drop_first=True)
print(f'After encoding: {df_fe.shape[1]} features')


## 4. Date-Based Features


In [ ]:
dates = pd.date_range('2020-01-01', periods=len(df_fe), freq='D')
df_fe['listing_year'] = dates.year
df_fe['listing_month'] = dates.month
df_fe['listing_dayofweek'] = dates.dayofweek
df_fe['listing_quarter'] = dates.quarter
df_fe['is_weekend'] = (dates.dayofweek >= 5).astype(int)
print('Date features created!')


## 5. Aggregate / Group Features


In [ ]:
# Average price by bedroom count
bedroom_avg_price = df.groupby('bedrooms')['price'].transform('mean')
df_fe['price_vs_bedroom_avg'] = df_fe['price'] - bedroom_avg_price

# Count of similar properties
df_fe['same_location_count'] = df.groupby('location').transform('size')
print('Aggregate features created!')


## 6. Measure the Impact


In [ ]:
target = 'price'
feature_cols = [c for c in df_fe.columns if c not in [target, 'log_price']]

X_raw = pd.get_dummies(df.drop(target, axis=1), columns=['location'], drop_first=True)
X_eng = df_fe[feature_cols]
y = df[target]

rf_raw = RandomForestRegressor(n_estimators=100, random_state=42)
rf_eng = RandomForestRegressor(n_estimators=100, random_state=42)

score_raw = cross_val_score(rf_raw, X_raw, y, cv=5, scoring='r2').mean()
score_eng = cross_val_score(rf_eng, X_eng, y, cv=5, scoring='r2').mean()

print('=== Feature Engineering Impact ===')
print(f'Raw features R²:        {score_raw:.4f}')
print(f'Engineered features R²: {score_eng:.4f}')
print(f'Improvement:            {(score_eng - score_raw):.4f}')
print(f'Feature count:          {X_raw.shape[1]} -> {X_eng.shape[1]}')
